# 00 — API Enumeration

Smoke test for `code/core/collective_cost.py`. Iterates every primitive at the
canonical regression anchor (N=512, M=16 MB, α=0.5 μs, BW=900 GB/s) and reproduces the ladders from `04_in_network_collectives.md` §3.1 and `05_contention_and_congestion.md` §6.1.

Treat this as a quick visual cross-section of the API. Trade-off sweeps (varying M, G, η, topology shape) live in subsequent notebooks.

## Setup

In [1]:
import sys
from pathlib import Path

# Make `code/` importable so we can do `from core...` and `from util...`.
REPO = Path.cwd().resolve()
while REPO.name != 'collective_comm' and REPO.parent != REPO:
    REPO = REPO.parent
CODE = REPO / 'code'
if str(CODE) not in sys.path:
    sys.path.insert(0, str(CODE))

import pandas as pd

from core import collective_cost as cc
from util import Anchors, ETA_PROFILES, PRIMITIVES, run_all, to_us

pd.set_option('display.max_rows', 100)
pd.set_option('display.float_format', lambda x: f'{x:10.3f}')
print(f'imports OK  (cwd={Path.cwd()})')
print(f'primitives registered: {len(PRIMITIVES)}')

imports OK  (cwd=/Users/jasonlu/Projects/ai.cluster/collective_comm/code/notebooks)
primitives registered: 35


## Anchor values

Defaults match the canonical regression anchor used throughout `documentation/modeling/`. Override fields on the `Anchors` dataclass to sweep.

In [2]:
anchors = Anchors()
for f in anchors.__dataclass_fields__:
    print(f'  {f:14s} = {getattr(anchors, f)!r}')

  M              = 16000000.0
  G              = 512
  alpha          = 5e-07
  bw             = 900000000000.0
  dims           = (8, 8, 8)
  L              = 8
  alpha_inner    = 5e-07
  bw_inner       = 900000000000.0
  alpha_outer    = 1e-06
  bw_outer       = 50000000000.0
  N_hier         = 512


## Full API enumeration

One row per primitive at default flags. Pipelined / mesh variants are listed as separate rows so the table reads as a cross-section. The `notes` column calls out non-default flags.

In [3]:
rows = run_all(anchors)
df = pd.DataFrame(rows)
df = df[['group', 'name', 't_us', 'notes', 'doc_ref']]
df

,group,name,t_us,notes,doc_ref
0,P2P,p2p_hop,18.278,,01_collective_algorithms.md §8
1,BC,ring_broadcast,9339.944,,01_collective_algorithms.md §3.1
2,BC,ring_broadcast (pipelined),273.278,P→P*,01_collective_algorithms.md §3.1 / App. C
3,BC,tree_broadcast,164.500,,01_collective_algorithms.md §3.2
4,BC,tree_broadcast (pipelined),22.278,P→P*,01_collective_algorithms.md §3.2 / App. C
5,BC,inc_broadcast,18.778,,04_in_network_collectives.md §1.2
6,BC,torus_broadcast,219.333,,02_topology_mapping.md §3.2
7,BC,torus_broadcast (mesh),383.833,open-line per dim,02_topology_mapping.md §4.2
8,Reduce,ring_reduce,9339.944,,01_collective_algorithms.md §4.1
9,Reduce,tree_reduce,164.500,,01_collective_algorithms.md §4.2


### Per-group view

Same data, sorted within each collective group by ascending time. Lets the best algorithm at this anchor jump to the top.

In [4]:
for g, sub in df.groupby('group', sort=False):
    print(f'\n=== {g} ===')
    print(sub.sort_values('t_us').to_string(index=False))


=== P2P ===
group    name       t_us notes                        doc_ref
  P2P p2p_hop     18.278       01_collective_algorithms.md §8

=== BC ===
group                       name       t_us             notes                                   doc_ref
   BC              inc_broadcast     18.778                           04_in_network_collectives.md §1.2
   BC tree_broadcast (pipelined)     22.278              P→P* 01_collective_algorithms.md §3.2 / App. C
   BC             tree_broadcast    164.500                            01_collective_algorithms.md §3.2
   BC            torus_broadcast    219.333                                 02_topology_mapping.md §3.2
   BC ring_broadcast (pipelined)    273.278              P→P* 01_collective_algorithms.md §3.1 / App. C
   BC     torus_broadcast (mesh)    383.833 open-line per dim               02_topology_mapping.md §4.2
   BC             ring_broadcast   9339.944                            01_collective_algorithms.md §3.1

=== Reduce ===
 gr

## Regression anchors — `04_in_network_collectives.md` §3.1 (ideal)

Targets at N=512, M=16 MB, α=0.5 μs, BW=900 GB/s:

| Algorithm | Target |
|---|---|
| star + ring AR | 546 μs |
| star + DBT (n_β=2 dual-touch) | 45 μs |
| hyp. 512-port star + NVLS | 18.8 μs |
| torus 8³ dim-decomp ring AR | 57 μs |

The DBT target uses the doc's `n_β=2` dual-touch ceiling (every byte crosses each endpoint link twice in the realized schedule), which is neither `pipelined=False` nor `pipelined=True` of `tree_all_reduce` — we compute it inline.

In [5]:
import math

M, G, alpha, bw = anchors.M, anchors.G, anchors.alpha, anchors.bw
log_G = math.ceil(math.log2(G))

ladder_ideal = [
    ('star + ring AR',
     to_us(cc.ring_all_reduce(M, G, alpha, bw)),                         546.0),
    ('star + DBT (n_β=2 dual-touch)',
     to_us(2 * log_G * alpha + 2.0 * (M / bw)),                          45.0),
    ('hyp. 512-port star + NVLS',
     to_us(cc.inc_all_reduce(M, alpha, bw)),                             18.8),
    ('torus 8³ dim-decomp ring AR',
     to_us(cc.torus_all_reduce(M, anchors.dims, alpha, bw)),             57.0),
]
ideal_df = pd.DataFrame(ladder_ideal, columns=['algorithm', 't_us', 'target_us'])
ideal_df['delta'] = ideal_df['t_us'] - ideal_df['target_us']
ideal_df

,algorithm,t_us,target_us,delta
0,star + ring AR,546.486,546.000,0.486
1,star + DBT (n_β=2 dual-touch),44.556,45.000,-0.444
2,hyp. 512-port star + NVLS,18.778,18.800,-0.022
3,torus 8³ dim-decomp ring AR,56.486,57.000,-0.514


## Regression anchors — `05_contention_and_congestion.md` §6.1 (realistic η)

Same N=512, M=16 MB anchor under the calibration profile from `05_contention_and_congestion.md` §4.1.

| Algorithm | Profile | Target |
|---|---|---|
| star + NVLS | nvls (η_β=0.52) | 35 μs |
| star + DBT (dual-touch) | crossbar (η_β=0.80) | 53 μs |
| torus 8³ dim-decomp ring AR | torus (η_α=1.20, η_β=0.60) | 84 μs |

In [6]:
from core.collective_cost import apply_eta

def eta_dbt_dual_touch(M, G, alpha, bw, eta_alpha, eta_beta):
    """DBT with the n_β=2 dual-touch BW form, η-discounted."""
    a, b = apply_eta(alpha, bw, eta_alpha, eta_beta)
    log_G = math.ceil(math.log2(G))
    return 2 * log_G * a + 2.0 * (M / b)

ladder_real = []
for label, fn, profile, target in [
    ('star + NVLS',
     lambda M, G, a, b: cc.inc_all_reduce(M, a, b),
     'nvls',     35.0),
    ('star + DBT (dual-touch)',
     eta_dbt_dual_touch,
     'crossbar', 53.0),
    ('torus 8³ dim-decomp ring AR',
     lambda M, G, a, b: cc.torus_all_reduce(M, anchors.dims, a, b),
     'torus',    84.0),
]:
    eta_a, eta_b = ETA_PROFILES[profile]
    if fn is eta_dbt_dual_touch:
        t_us = to_us(eta_dbt_dual_touch(M, G, alpha, bw, eta_a, eta_b))
    else:
        a, b = apply_eta(alpha, bw, eta_a, eta_b)
        t_us = to_us(fn(M, G, a, b))
    ladder_real.append((label, profile, t_us, target))

real_df = pd.DataFrame(ladder_real, columns=['algorithm', 'profile', 't_us', 'target_us'])
real_df['delta'] = real_df['t_us'] - real_df['target_us']
real_df

,algorithm,profile,t_us,target_us,delta
0,star + NVLS,nvls,35.188,35.000,0.188
1,star + DBT (dual-touch),crossbar,53.444,53.000,0.444
2,torus 8³ dim-decomp ring AR,torus,84.344,84.000,0.344


## Hierarchical sanity — `03_hierarchical_topologies.md` §2.1

NVL72-style (inner ring at NVLink-class α/BW) + IB-class outer scale-out. Walks through the RS → outer sub-AR → AG composition explicitly so the telescoped payload `M·L/N` is visible.

In [7]:
N = anchors.N_hier
L = anchors.L
n_inner = N // L
M_outer = anchors.M * L / N    # telescoped payload

rs_inner = cc.ring_reduce_scatter(anchors.M, n_inner,
                                  anchors.alpha_inner, anchors.bw_inner)
ar_outer = cc.ring_all_reduce(M_outer, L,
                              anchors.alpha_outer, anchors.bw_outer)
ag_inner = cc.ring_all_gather(anchors.M, n_inner,
                              anchors.alpha_inner, anchors.bw_inner)

phases = pd.DataFrame([
    {'phase': 'inner RS',       'payload_MB': anchors.M / 1e6,  'group': n_inner, 't_us': to_us(rs_inner)},
    {'phase': 'outer sub-AR',   'payload_MB': M_outer / 1e6,    'group': L,       't_us': to_us(ar_outer)},
    {'phase': 'inner AG',       'payload_MB': anchors.M / 1e6,  'group': n_inner, 't_us': to_us(ag_inner)},
    {'phase': 'TOTAL',          'payload_MB': float('nan'),     'group': N,       't_us': to_us(rs_inner + ar_outer + ag_inner)},
])
phases

,phase,payload_MB,group,t_us
0,inner RS,16.000,64,49.000
1,outer sub-AR,0.250,8,22.750
2,inner AG,16.000,64,49.000
3,TOTAL,NaN,512,120.750


## What's next

- `01_*.ipynb` — sweep M (small payload α-bound vs large-M BW-bound regime crossover).
- `02_*.ipynb` — sweep G / topology shape (star vs DBT vs torus dim-decomp at fixed M).
- `03_*.ipynb` — η contention sensitivity (which algorithms are robust to bad calibration).
- `04_*.ipynb` — hierarchical partitioning sweep across L (TP/EP/PP/DP allocation).

The shared `Anchors` / `ETA_PROFILES` / `PRIMITIVES` registry in `code/util/` is the substrate for those — extend the registry or override `Anchors` fields, then call `run_all` again.